# Content-Based Movie Recommender
A simple recommender built with pandas on the MovieLens `movies.csv` file.

**Idea:** build a *taste profile* for a user from the genres of the movies they rated (ratings are centered around a neutral midpoint, so low ratings count negatively), then rank all movies by the cosine similarity between their genres and that profile. Movies the user already rated are removed.

## 1. Load the data

In [131]:
import pandas as pd

In [132]:
df = pd.read_csv('movies.csv')

In [133]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [134]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34208 entries, 0 to 34207
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  34208 non-null  int64
 1   title    34208 non-null  str  
 2   genres   34208 non-null  str  
dtypes: int64(1), str(2)
memory usage: 801.9 KB


In [135]:
df.shape

(34208, 3)

In [136]:
df.duplicated().sum()

np.int64(0)

In [137]:
df.columns

Index(['movieId', 'title', 'genres'], dtype='str')

## 2. Clean the data
Split the year out of the title, e.g. `Toy Story (1995)` -> `Toy Story` and `1995`.

In [138]:
df["year"] = df["title"].str.extract(r"\((\d{4})\)").astype("Int64")
df["title"]= df["title"].str.replace(r"\((\d{4})\)" , "" ,regex=True).str.strip()

In [139]:
df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


## 3. One-hot encode the genres
Each genre becomes a 0/1 column. Movies with no genre are removed.

In [142]:
dummies = df["genres"].str.get_dummies(sep="|")
df = pd.concat([df , dummies] , axis=1)

In [143]:
print(df.shape)
df= df[df["(no genres listed)"] == 0]
df.shape

(34208, 24)


(33063, 24)

In [144]:
df = df.drop(["genres" ,"(no genres listed)" ] , axis=1)
df.head()

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,1995,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji,1995,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


Remove movies whose year is missing.

In [145]:
df = df.dropna()
df.isna().sum()

movieId        0
title          0
year           0
Action         0
Adventure      0
Animation      0
Children       0
Comedy         0
Crime          0
Documentary    0
Drama          0
Fantasy        0
Film-Noir      0
Horror         0
IMAX           0
Musical        0
Mystery        0
Romance        0
Sci-Fi         0
Thriller       0
War            0
Western        0
dtype: int64

## 4. Example user ratings
A small made-up user with ratings on a 0-10 scale.

In [146]:
user = pd.DataFrame({
    "userId": [1, 1, 1, 1, 1, 1],
    "title": [
        "Toy Story (1995)",
        "Heat (1995)",
        "Jumanji (1995)",
        "Interstellar (2014)",
        "Se7en (1995)",
        "Babe (1995)"
    ],
    "rating": [8.5, 3.0, 6.5, 10.0, 5.0, 3.5]
})
user


,userId,title,rating
0,1,Toy Story (1995),8.5
1,1,Heat (1995),3.0
2,1,Jumanji (1995),6.5
3,1,Interstellar (2014),10.0
4,1,Se7en (1995),5.0
5,1,Babe (1995),3.5


In [147]:
user = user.drop([ "userId"], axis=1)
user["year"] = user.title.str.extract(r"\((\d{4})\)").astype("Int64")
user["title"] = user.title.str.replace(r"\((\d{4})\)" , "" ,regex=True).str.strip()
user


,title,rating,year
0,Toy Story,8.5,1995
1,Heat,3.0,1995
2,Jumanji,6.5,1995
3,Interstellar,10.0,2014
4,Se7en,5.0,1995
5,Babe,3.5,1995


In [148]:
df[df["title"].str.contains("seven|se7en" , case=False , regex=True) & (df["year"] == 1995)]

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
46,47,Seven (a.k.a. Se7en),1995,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
24588,115607,"Seventh Brother, The (A hetedik testvér)",1995,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Se7en is stored as `Seven (a.k.a. Se7en)` in the dataset, so a merge on the title would silently drop it. We add a row with the exact dataset title.

In [149]:
row = pd.DataFrame(df[df["movieId"]==47][["title","year"]]).copy()
row["rating"] = 5.0
user = pd.concat([user , row] , ignore_index=True)
user

,title,rating,year
0,Toy Story,8.5,1995
1,Heat,3.0,1995
2,Jumanji,6.5,1995
3,Interstellar,10.0,2014
4,Se7en,5.0,1995
5,Babe,3.5,1995
6,Seven (a.k.a. Se7en),5.0,1995


## 5. Center the ratings
Subtract a neutral midpoint: above it a rating is positive, below it negative, and the further away, the stronger the effect.

In [150]:
midpoint = 5   # neutral rating on a 0-10 scale (use 2.75 for MovieLens' 0.5-5 scale)

# Run this cell only once: it changes "rating" in place. Use Restart & Run All to redo it.
user["rating"] = user["rating"] - midpoint

Remove the old `Se7en` row, which never matches the dataset title.

In [151]:
user = user[user["title"] != "Se7en"]

## 6. Match the user's movies with the dataset

In [152]:
user_movies = df.merge(user , on=["title" , "year"] ,how='inner')
user_movies

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,rating
0,1,Toy Story,1995,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,3.5
1,2,Jumanji,1995,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1.5
2,6,Heat,1995,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,-2.0
3,34,Babe,1995,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,-1.5
4,47,Seven (a.k.a. Se7en),1995,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.0
5,109487,Interstellar,2014,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,5.0


In [153]:
user_movies= user_movies.drop(["movieId" , "title" ,"year" ], axis=1)

In [154]:
user_rate = user_movies["rating"].tolist()
user_movies = user_movies.drop("rating" , axis=1)

## 7. Build the user's taste profile
Multiply each movie's genres by its centered rating, then sum per genre.

In [155]:
genre_rating = user_movies.mul(user_rate , axis=0)
genre_rating_sum = genre_rating.sum()
genre_rating_sum

Action        -2.0
Adventure      5.0
Animation      3.5
Children       3.5
Comedy         3.5
Crime         -2.0
Documentary    0.0
Drama         -1.5
Fantasy        5.0
Film-Noir      0.0
Horror         0.0
IMAX           5.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         5.0
Thriller      -2.0
War            0.0
Western        0.0
dtype: float64

## 8. Prepare the genre matrix of all movies

In [156]:
df_genres = df.drop([ "title" , "year"] , axis=1)
print(df_genres["movieId"].is_unique)

df_genres = df_genres.set_index("movieId")

True


## 9. Score every movie with cosine similarity

In [159]:
profile = genre_rating_sum                  # taste vector: one number per genre

dot = df_genres.dot(profile)                # per movie: sum of (genre flag * taste value)
n_genres = df_genres.sum(axis=1)            # number of genres of each movie
movie_norm = n_genres ** 0.5                # length of each movie's 0/1 genre vector
profile_norm = (profile ** 2).sum() ** 0.5  # length of the taste vector

# Cosine similarity = dot / (|movie| * |profile|)
recomend = (dot / (movie_norm * profile_norm)).sort_values(ascending=False)
recomend.head(10)

movieId
32031     0.938129
673       0.847181
78637     0.847181
47124     0.847181
130520    0.847181
78499     0.847181
3054      0.800662
76093     0.800662
2054      0.800662
30793     0.800662
dtype: float64

## 10. Remove movies the user already rated

In [166]:
watched_ids = df.merge(user , on=["title" , "year"])["movieId"]
recomend = recomend[~recomend.index.isin(watched_ids)]

## 11. Final recommendations

In [162]:
titles = df.set_index("movieId")["title"]
result = recomend.head(10).to_frame("score")
result["title"] = titles
result

,score,title
movieId,,
32031,0.938129,Robots
673,0.847181,Space Jam
78637,0.847181,Shrek Forever After (a.k.a. Shrek: The Final C...
47124,0.847181,"Ant Bully, The"
130520,0.847181,Home
78499,0.847181,Toy Story 3
3054,0.800662,Pokémon: The First Movie
76093,0.800662,How to Train Your Dragon
2054,0.800662,"Honey, I Shrunk the Kids"
